# 第6讲 —— 空间异质性与校准 (Spatial Heterogeneity and Calibration)

**异质性主体宏观经济学的计算方法 (Computational Methods for Heterogeneous-Agent Macro)**

孙杰

### 环境配置

激活项目环境,加载 `HouseholdStages` 以及 `Printf` / `Plots`。

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using HouseholdStages
using Printf
using Plots

## 1 · 空间模型设置

三个地区 `loc1`、`loc2`、`loc3`,生产率满足 $A_1 > A_2 > A_3$。每个地区生产单一的、无成本可贸易、完全替代的计价品(numeraire good),采用 Cobb–Douglas 技术
$$Y_j = A_j K_j^{\alpha} L_j^{1-\alpha}.$$
资本在地区之间以及与世界其他地区之间自由流动,因此本国在世界债券市场上是小国:**$r$ 是外生的**。在国内,资本边际产出(MPK)在地区间的均等化决定了给定 $(r, A_j)$ 下的 $K_j / L_j$,工资由此满足
$$w_j = (1-\alpha)\, A_j\, \big(\tfrac{\alpha A_j}{r+\delta}\big)^{\alpha/(1-\alpha)}.$$

**与 L04 的联系。** 我们采取与 L04 不同的方式来闭合模型:$r$ 由世界债券市场决定,因此不需要 $K$ 的 tatonnement 迭代。外层循环是偏好移位项(preference shifters)$\alpha = (\alpha_1, \alpha_2, \alpha_3)$ 的**校准 (calibration)** —— 见 §5 —— 它取代市场出清成为均衡条件。

家庭仅在**财富**和**地区**上具有异质性(没有特异收入冲击)。在一个时期内:

3. **消费**

链条为
$$\text{Migration} \circ \text{WealthChange} \circ \text{ConsumptionSavings}.$$

### 参数与目标份额

标准宏观校准。世界利率为 $r = 0.03$。生产率为 $(1.20, 1.00, 0.85)$ —— `loc1` 是高工资地区。迁移成本(migration cost)$C[i, j]$ 是对称的,相邻地区之间的迁移比远距离地区之间更便宜。偏好移位项以 $\alpha = (0, 0, 0)$ 作为校准循环的初始猜测 —— 根据归一化 $\alpha_1 \equiv 0$,只有 $(\alpha_2, \alpha_3)$ 是自由参数。

目标人口份额 `s_data = (0.30, 0.30, 0.40)` 是合成的:*数据*显示有 40% 的家庭居住在 `loc3`(生产率最低的地区),因此我们需要一个正的偏好移位项作用于 `loc3` 才能合理化它们。§5 中校准的全部要义就是恢复这个移位项。

In [ ]:
@kwdef struct SpatialParams3
    β::Float64 = 0.96
    σ::Float64 = 1.5
    α::Float64 = 0.36
    δ::Float64 = 0.08
    r::Float64 = 0.03                          # 世界利率(外生)
    A::NTuple{3,Float64} = (1.20, 1.00, 0.85)  # 生产率
    # 迁移成本矩阵 C[i, j](起点 → 终点)。
    C_base::Matrix{Float64} = [0.0 0.5 1.0;
                               0.5 0.0 0.5;
                               1.0 0.5 0.0]
    η_logit::Float64 = 1.5                     # Gumbel 尺度
    # (α₂, α₃) 校准迭代的初始猜测;α₁ ≡ 0。
    α_init::NTuple{2,Float64} = (0.0, 0.0)
    N_w::Int       = 250
    w_min::Float64 = 0.0
    w_max::Float64 = 25.0
end
Base.Broadcast.broadcastable(p::SpatialParams3) = Ref(p)

const LOC_NAMES = (:loc1, :loc2, :loc3)
loc_idx(sym::Symbol) = findfirst(==(sym), LOC_NAMES)

const s_data = [0.30, 0.30, 0.40]
p = SpatialParams3()
@printf "β=%.2f, σ=%.2f, α=%.2f, δ=%.2f, r=%.4f\n" p.β p.σ p.α p.δ p.r
@printf "A = (%.2f, %.2f, %.2f);  η_logit = %.2f\n" p.A[1] p.A[2] p.A[3] p.η_logit
println("target shares = ", s_data)

## 2 · 由世界利率得出工资

`spatial_wage(r, A_j, p)` 实现 $w_j = (1-\alpha) A_j (\alpha A_j / (r+\delta))^{\alpha/(1-\alpha)}$。我们把它在 `p.A` 上广播,得到长度为 3 的工资向量 `w_vec`。我们传给家庭模块(household block)的 env 将携带 `w_vec` 和当前校准迭代中的 $\alpha$(同样长度为 3)—— 这要等到构建好链条后才能完成。

In [ ]:
function spatial_wage(r, A_j, p)
    return (1 - p.α) * A_j * (p.α * A_j / (r + p.δ))^(p.α / (1 - p.α))
end

w_vec = spatial_wage.(p.r, p.A, p)
@printf "Wages: w = (%.4f, %.4f, %.4f)\n" w_vec[1] w_vec[2] w_vec[3]
@printf "Ratio w[1]/w[3] = %.3f (productivity ratio A[1]/A[3] = %.3f)\n" w_vec[1]/w_vec[3] p.A[1]/p.A[3]

## 3 · 家庭模块

三个阶段。迁移阶段携带一个**便利度闭包 (amenity closure)**,从 `env` 中读取偏好移位项向量 $\alpha$:
$$\text{amenity}(j;\, \texttt{env}) = \texttt{env.}\alpha[j],$$
约定 $\alpha_1 \equiv 0$ 钉住地区效用(只有差值可识别)。库在每次反向传递中只把长度为 3 的便利度向量物化一次,因此热路径不会比静态向量便利度更慢。

$\alpha$ 是 env 上的数据,不是 Spec 上的数据。§5 中的校准循环将通过**重建 env** 来改变 $\alpha$ —— 没有对 Spec 的就地修改。

In [ ]:
_u_crra(c, ::Val{1})           = log(c)
_u_crra(c, ::Val{σv}) where σv = (c^(1 - σv)) / (1 - σv)
u_crra(c, valσ::Val) = c < 0 ? -Inf : _u_crra(c, valσ)

function spatial_household(p::SpatialParams3)
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:location, categorical(collect(LOC_NAMES))),
    )

    # 便利度闭包:在目的地索引 env.α[j]。
    amenity = (dest; env) -> env.α[loc_idx(dest)]

    migration = MigrationStage(layout; migration_cost=p.C_base, amenity, ε=p.η_logit)

    income = WealthChangeStage(layout; wealth_post = function (cell; env)
        return (1 + env.r) * cell.wealth + env.w[loc_idx(cell.location)]
    end)

    savings = ConsumptionSavingsStage(layout; β=p.β, utility=(cell, c; env) -> u_crra(c, Val(p.σ)), monotone_search=:divide_conquer)

    hh = migration ∘ income ∘ savings

    return define_moments!(hh;
        K_total = at_end(integrand=(cell; env) -> cell.wealth, reduce=sum),
        L       = at_end(integrand=(cell; env) -> Float64[cell.location == j for j in LOC_NAMES], reduce=sum),
    )
end

hh = spatial_household(p)
dims = layout_size(first(hh.spec.stages).input_layout)
@printf "Layout: wealth %d × location %d = %d cells\n" dims[1] dims[2] prod(dims)

### 3.1 `hh` 里有什么?

简单看一下迁移阶段的 Spec 和 Buffer。Spec 携带成本矩阵和便利度*闭包*;Buffer 携带每个 cell 的选择概率张量 `choice_prob` —— 迁移阶段的"记忆" —— 在每次 `backward!` 时由对数求和指数(log-sum-exp)重新填充。

In [ ]:
mig = hh.spec.stages[1]
mig_buf = hh.buffer.stages[1]
@show typeof(mig)
@show typeof(mig.amenity)                       # 闭包:(dest; env) -> Real
@show size(mig_buf.kernel.choice_prob)          # (N_w, 3, 3):每个 (wealth, origin),每个 destination

## 4 · 在 $\alpha = 0$ 下求解家庭

在初始猜测 $\alpha = 0$ 下的**演示运行** —— *在*校准循环启动之前。选择概率仅依赖于工资和迁移成本。`loc1` 的工资最高;我们预期它会吸引最大的人口份额。§5 中的校准随后会调整 $\alpha$ 以匹配数据。

In [ ]:
function solve_household(hh, p, α; V_init=nothing, Λ_init=nothing)
    env = make_env(hh; r=p.r, w=spatial_wage.(p.r, p.A, p), α)

    (;V, Λ, moments, history) = solve_steady_state_given_env!(hh, env; V_init, Λ_init)
    shares = moments.L ./ sum(moments.L)

    return (; V, Λ, env, moments, shares,
              vfi_iters=history.vfi_iters, lambda_iters=history.lambda_iters)
end

out0 = solve_household(hh, p, [0.0, 0.0, 0.0])
@printf "mass conservation: ΣΛ = %.10f\n" sum(out0.Λ)
@printf "VFI %d iters, Λ %d iters\n" out0.vfi_iters out0.lambda_iters
@printf "Population shares (uncalibrated): (%.3f, %.3f, %.3f)\n" out0.shares[1] out0.shares[2] out0.shares[3]
@printf "Total wealth K_total = %.4f\n" out0.moments.K_total

相对于数据,高生产率地区 `loc1` 人口过多;`loc3` 人口不足。有某种东西让人们留在 `loc3`,而模型 —— 在 $\alpha = 0$ 下 —— 看不到。这正是我们将要校准的东西。

In [ ]:
xs = 1:3
plt_init = plot(title="Uncalibrated vs target shares", ylabel="share",
                xticks=(xs, ["loc1", "loc2", "loc3"]),
                ylims=(0.0, 0.5), legend=:topright, size=(640, 360))
bar!(plt_init, xs .- 0.18, out0.shares; bar_width=0.30, label="model (α=0)")
bar!(plt_init, xs .+ 0.18, s_data;      bar_width=0.30, label="target (data)")

## 5 · 通过阻尼对数份额更新进行校准

仅有工资的模型无法匹配 `loc3` 的数据。我们需要偏好移位项 $(\alpha_2, \alpha_3)$ —— 回忆 $\alpha_1 \equiv 0$ —— 使得稳态人口份额等于 `s_data`。三步走:理解映射 $\alpha \to \text{shares}$,写出它的(近似)逆映射,然后运行循环。

### 5.1 映射 $\alpha \to$ 份额

直觉:更大的便利度 $\alpha_j$ 让目的地 $j$ 更具吸引力,因此稳态下更多家庭迁入。具体地,把 $\alpha_3$ 从 0 提高到 +0.5 应该把质量*移入* `loc3` 并*移出* `loc1` / `loc2`。让我们验证一下。

In [ ]:
out_bump = solve_household(hh, p, [0.0, 0.0, 0.5])
@printf "α₃ = 0.0:  shares = (%.3f, %.3f, %.3f)\n" out0.shares[1] out0.shares[2] out0.shares[3]
@printf "α₃ = 0.5:  shares = (%.3f, %.3f, %.3f)\n" out_bump.shares[1] out_bump.shares[2] out_bump.shares[3]
@printf "Δ on loc3: %+.3f\n" out_bump.shares[3] - out0.shares[3]

### 5.2 反演映射 —— 阻尼对数份额更新

在**静态** logit 模型中,更大的便利度 $\alpha_j$ 会让质量按 $\log s_j$ 的比例向 $j$ 转移。更新式
$$\alpha_j \;\leftarrow\; \alpha_j \;+\; \eta_{\text{logit}} \cdot \bigl(\log s_j^{\text{data}} - \log s_j^{\text{model}}\bigr)$$
是朝着对数份额缺口不动点的一个阻尼步。在我们的**动态**模型中,$V_j$ 也通过未来迁移价值依赖于 $\alpha$,所以这个静态公式不再是精确反演 —— 但相同的更新仍然是一个有用、经验上保守的不动点迭代。**代码中:** 乘以 `update_speed` $\in (0, 1]$ 以确保安全,丢弃 $\alpha_1$ 的梯度以强制归一化,然后让它运行。

In [ ]:
function calibrate_shifters!(hh, p, s_data; verbosity=1)
    update_speed = 0.1
    tol     = 5e-3
    maxiter = 60

    # α = (α₁, α₂, α₃),其中由归一化 α₁ ≡ 0。
    α = Float64[0.0, p.α_init[1], p.α_init[2]]
    V, Λ = nothing, nothing
    α_history   = Vector{Float64}[copy(α)]
    gap_history = Vector{Float64}[]
    last_out    = nothing
    iters       = 0
    converged   = false

    while iters < maxiter
        out = solve_household(hh, p, α; V_init=V, Λ_init=Λ)
        last_out = out
        (;V, Λ) = out

        gap = log.(s_data) .- log.(max.(out.shares, 1e-8))
        push!(gap_history, copy(gap))
        iters += 1

        verbosity > 0 && @printf("  iter %2d: shares = (%.3f, %.3f, %.3f); α = (%.3f, %.3f, %.3f); ‖gap‖∞ = %.4f\n",
            iters, out.shares[1], out.shares[2], out.shares[3],
            α[1], α[2], α[3], maximum(abs.(gap)))

        if maximum(abs.(gap)) < tol
            converged = true
            break
        end

        # 阻尼对数份额更新:αⱼ ← αⱼ + s · η · (log sᵈᵃᵗᵃ − log sᵐᵒᵈᵉˡ);重新归一化 α[1] = 0。
        α .+= update_speed * p.η_logit .* gap
        α .-= α[1]
        push!(α_history, copy(α))
    end

    return (; α, iters, converged, α_history, gap_history,
              shares=last_out.shares, V=last_out.V, Λ=last_out.Λ,
              env=last_out.env, moments=last_out.moments)
end

### 运行校准

循环从 $\alpha = 0$ 开始(此时 `loc3` 人口不足),迈出一个阻尼对数份额步,重新求解家庭问题,如此往复。收敛是几何速度的。

In [ ]:
calib = calibrate_shifters!(hh, p, s_data; verbosity=1)
@printf "\nFinal α = (%.4f, %.4f, %.4f); converged = %s in %d iterations\n" calib.α[1] calib.α[2] calib.α[3] calib.converged calib.iters
@printf "Final shares: (%.3f, %.3f, %.3f) vs target (%.3f, %.3f, %.3f)\n" calib.shares[1] calib.shares[2] calib.shares[3] s_data[1] s_data[2] s_data[3]

### 收敛诊断

对数份额缺口以几何速度坍缩;移位项的轨迹单调地朝不动点上升。

In [ ]:
αs   = hcat(calib.α_history...)
gaps = hcat(calib.gap_history...)

plt_α   = plot(1:size(αs, 2), αs',
               label=["α[1]" "α[2]" "α[3]"], marker=:circle, linewidth=2,
               xlabel="calibration iteration", ylabel="α_j",
               title="Preference shifter trajectory")

plt_gap = plot(1:size(gaps, 2), maximum.(abs, eachcol(gaps)),
               marker=:circle, linewidth=2, yaxis=:log,
               xlabel="calibration iteration", ylabel="max gap (log scale)",
               title="‖log s_data − log s_model‖∞", legend=false)

plot(plt_α, plt_gap; layout=(1, 2), size=(900, 360))

### 最终份额:未校准、已校准、目标

In [ ]:
xs = 1:3
plt_final = plot(title="Population shares", ylabel="share",
                 xticks=(xs, ["loc1", "loc2", "loc3"]),
                 ylims=(0.0, 0.5), legend=:topright, size=(640, 360))
bar!(plt_final, xs .- 0.25, out0.shares;  bar_width=0.22, label="uncalibrated (α=0)")
bar!(plt_final, xs,         calib.shares; bar_width=0.22, label="calibrated")
bar!(plt_final, xs .+ 0.25, s_data;       bar_width=0.22, label="target (data)")

## 6 · 解读校准后的移位项

- $\alpha_1 = 0$ 由归一化决定。数据中 `loc1` 的 30% 份额*完全可由*其生产率优势解释;它不需要任何便利度加成。
- $\alpha_2 \approx 0$(约为 $-0.013$)。`loc2` 的 30% 份额也大致与仅工资模型一致。
- $\alpha_3 \approx 0.75$。尽管 `loc3` 的工资*最低*,却容纳了 40% 的家庭。数据告诉我们,每期约有 0.75 单位的效用属于工资模型所遗漏的"便利度"或不可观察偏好。这正是间接识别(indirect identification)所能给我们带来的推断:一个我们无法直接观察的量的值。

## 7 · 进一步观察:按地区的财富分布

§5 说 $\alpha$ 在地区之间移动*质量*。对偶问题 —— 给定地区,财富如何分布?—— 才是工资差异显现之处。在没有收入异质性的情况下,唯一的缓冲性储蓄(buffer-stock)动机是未来可能迁移到不同工资地区的*可能性*,因此我们预期条件于地区的财富分布会很紧、且其均值跟随当地工资。

每个地区三个汇总统计量:

In [ ]:
"""CLAUDE
计算离散分布 `(values, mass)` 的 Gini 系数,其中 `mass` 不必和为 1
(内部会归一化)。值按升序排序;Lorenz 曲线积分采用对已排序
(累积人口, 累积财富) 对的梯形规则计算。
"""
function _gini(values::AbstractVector, mass::AbstractVector)
    total = sum(mass)
    total ≤ 0 && return NaN
    perm = sortperm(values)
    v    = values[perm]
    p    = mass[perm] ./ total
    cumpop    = vcat(0.0, cumsum(p))
    cumwealth = vcat(0.0, cumsum(v .* p) ./ sum(v .* p))
    area_under_lorenz = sum(diff(cumpop) .* (cumwealth[2:end] .+ cumwealth[1:end-1]) ./ 2)
    return 1 - 2 * area_under_lorenz
end

wgrid = first(hh.spec.stages).input_layout.axes[1].kind.grid
Λ_ss  = calib.Λ
w_vec = spatial_wage.(p.r, p.A, p)

println("location |  wage  |  share | mean w  | Pr(w=0) |  Gini")
println("---------|--------|--------|---------|---------|--------")
for j in 1:3
    mass_j  = @view Λ_ss[:, j]
    share_j = sum(mass_j)
    mean_w  = sum(wgrid .* mass_j) / share_j
    p_atbc  = mass_j[1] / share_j
    gini_j  = _gini(wgrid, mass_j)
    @printf "%-8s | %.4f | %.3f  | %.4f  |  %.3f  | %.3f\n" string(LOC_NAMES[j]) w_vec[j] share_j mean_w p_atbc gini_j
end

**解读这张表。** `loc1` 是高工资地区:根据 §5 的校准,它只容纳 30% 的家庭,但每个家庭都更富有(均值 $b$ 最高,受约束的质量最小)。`loc3` 在*最低*工资下承载了 40% 的人口 —— 便利度 $\alpha_3$ 把家庭拉进来,而由于其中一些家庭储蓄缓冲较低、工资也无助于他们积累缓冲,受约束的质量更大,Gini 系数也更高。两个渠道 —— $\alpha$ 移动*质量*,工资移动*条件于地区的财富* —— 在同一幅图中可以分别看到。

### 7.1 按地区的条件财富分布

边际分布 $\Lambda[:, j]$ 与地区 $j$ 的*人口*成比例,这使得朴素的图容易产生误导 —— `loc1` 较低的曲线会被误读为"该地区更穷",而它实际上只是人口更少。我们把每一列归一化以使其积分为 1,绘制给定地区下财富的*条件*分布。财富在对数轴上,因为网格是对数间隔的(靠近零处密,顶部稀疏)。

每条曲线积分为 1 —— 它们在*形状*上可直接比较,但在*水平*上不行。

In [ ]:
Λ_cond = Λ_ss ./ sum(Λ_ss; dims=1)   # 将每个地区列归一化以求和为 1
plot(wgrid, Λ_cond;
     labels=["loc1" "loc2" "loc3"],
     xlims=(wgrid[2], wgrid[end]),   # 跳过 wgrid[1] = 0 以适应对数坐标
     xscale=:log10,
     xlabel="wealth (log scale)", ylabel="conditional mass",
     title="Wealth | location  (each curve integrates to 1)",
     linewidth=2, size=(720, 360))

## 8 · 预告

空间模型的债券市场是外生的:我们从世界市场设定 $r$。**L07** 将把资产市场重新放回模型内部,并加入*总量*不确定性 —— 也就是 Krusell–Smith 问题。